# Modèle Multi-Tâche avec MobileNetV3 (Transfer Learning)

Ce notebook utilise **MobileNetV3** comme backbone pré-entraîné.

## Principe du Transfer Learning

```
Image d'entrée (128x128)
        │
        ▼
┌───────────────────────────────────┐
│   MobileNetV3 (Pré-entraîné)      │  ← Connaissance ImageNet
│   FROZEN (on ne touche pas)       │  ← 1000 classes apprises
│   = Extracteur de features        │
└───────────────────────────────────┘
        │
        ▼ Features extraites (1280 dim)
┌───────────────────────────────────┐
│   Nos couches personnalisées      │  ← ON ENTRAÎNE SEULEMENT ÇA
│   (Dense layers + Dropout)        │
└───────────────────────────────────┘
        │
    ┌───┴───┐───────┐
    ▼       ▼       ▼
  Age    Genre   Ethnicité
```

### Pourquoi MobileNet ?

- **Plus léger** : ~2.5M params vs ~4M (EfficientNet)
- **Plus rapide** sur mobile
- **Pré-entraîné** sur ImageNet (1.2M images, 1000 classes)
- Sait déjà reconnaître des formes, textures, visages basiques


## 1. Imports et Configuration


In [1]:
import os
import warnings

warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm.auto import tqdm
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from scipy import stats

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, callbacks
from tensorflow.keras.applications import MobileNetV3Large, MobileNetV3Small
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.optimizers.schedules import CosineDecay
from tensorflow.keras.regularizers import l2

# Configuration GPU
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU disponible: {gpus}")

# Seed
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.figsize": (12, 6), "axes.titlesize": 14})

print("Imports OK!")

TensorFlow version: 2.20.0
GPU disponible: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Imports OK!


## 2. Hyperparamètres


In [2]:
# Chemins
BASE_DIR = Path(".")
DATA_DIR = BASE_DIR / "data" / "UTKFace"
ARTIFACTS_DIR = BASE_DIR / "artifacts"
ARTIFACTS_DIR.mkdir(exist_ok=True)

# Hyperparamètres
IMG_SIZE = 128
BATCH_SIZE = 32

# Epochs (MobileNet converge plus vite car plus léger)
WARMUP_EPOCHS = 15
FINETUNE_EPOCHS = 40

# Learning rates
INITIAL_LR = 1e-3  # Plus élevé car MobileNet est plus stable
FINETUNE_LR = 1e-5

# Splits
VAL_SPLIT = 0.15
TEST_SPLIT = 0.15

# Classes
ETHNICITY_CLASSES = ["White", "Black", "Asian", "Indian", "Others"]
GENDER_CLASSES = ["Male", "Female"]
NUM_ETHNICITY_CLASSES = len(ETHNICITY_CLASSES)

# Choisir la variante MobileNet
USE_LARGE = True  # True = MobileNetV3Large (meilleur), False = Small (plus rapide)

print("Configuration MobileNet:")
print(f"  - Variante: MobileNetV3{'Large' if USE_LARGE else 'Small'}")
print(f"  - Image size: {IMG_SIZE}x{IMG_SIZE}")
print(f"  - Batch size: {BATCH_SIZE}")
print(f"  - Warmup epochs: {WARMUP_EPOCHS}")
print(f"  - Finetune epochs: {FINETUNE_EPOCHS}")

Configuration MobileNet:
  - Variante: MobileNetV3Large
  - Image size: 128x128
  - Batch size: 32
  - Warmup epochs: 15
  - Finetune epochs: 40


## 3. Chargement des Données


In [3]:
def parse_utkface_filename(filepath):
    """Parse le nom de fichier UTKFace."""
    filename = Path(filepath).stem
    parts = filename.split("_")

    if len(parts) >= 3:
        try:
            age = int(parts[0])
            gender = int(parts[1])
            ethnicity = int(parts[2])

            if 0 <= age <= 116 and gender in [0, 1] and 0 <= ethnicity <= 4:
                return {
                    "filepath": str(filepath),
                    "age": age,
                    "gender": gender,
                    "ethnicity": ethnicity,
                }
        except (ValueError, IndexError):
            pass
    return None


# Charger
image_paths = list(DATA_DIR.glob("*.jpg")) + list(DATA_DIR.glob("*.JPG"))
print(f"Images trouvées: {len(image_paths)}")

records = [parse_utkface_filename(path) for path in tqdm(image_paths, desc="Parsing")]
records = [r for r in records if r is not None]

df = pd.DataFrame(records)
print(f"Images valides: {len(df)}")

# Nettoyage outliers
z_scores = np.abs(stats.zscore(df["age"]))
df_clean = df[z_scores <= 3].copy()
print(f"Après nettoyage: {len(df_clean)}")

# Distribution
print(f"\nDistribution ethnicité:")
for i, name in enumerate(ETHNICITY_CLASSES):
    count = (df_clean["ethnicity"] == i).sum()
    print(f"  {name}: {count} ({count/len(df_clean)*100:.1f}%)")

Images trouvées: 23708


Parsing: 100%|██████████| 23708/23708 [00:00<00:00, 274422.28it/s]

Images valides: 23705
Après nettoyage: 23633

Distribution ethnicité:
  White: 10034 (42.5%)
  Black: 4520 (19.1%)
  Asian: 3415 (14.5%)
  Indian: 3972 (16.8%)
  Others: 1692 (7.2%)


## 4. Split et Poids de Classes


In [4]:
# Données
X = df_clean["filepath"].values
y_age = df_clean["age"].values.astype(np.float32)
y_gender = df_clean["gender"].values.astype(np.int32)
y_ethnicity = df_clean["ethnicity"].values.astype(np.int32)

# Split
(
    X_trainval,
    X_test,
    y_age_trainval,
    y_age_test,
    y_gender_trainval,
    y_gender_test,
    y_eth_trainval,
    y_eth_test,
) = train_test_split(
    X,
    y_age,
    y_gender,
    y_ethnicity,
    test_size=TEST_SPLIT,
    random_state=SEED,
    stratify=y_ethnicity,
)

val_ratio = VAL_SPLIT / (1 - TEST_SPLIT)
(
    X_train,
    X_val,
    y_age_train,
    y_age_val,
    y_gender_train,
    y_gender_val,
    y_eth_train,
    y_eth_val,
) = train_test_split(
    X_trainval,
    y_age_trainval,
    y_gender_trainval,
    y_eth_trainval,
    test_size=val_ratio,
    random_state=SEED,
    stratify=y_eth_trainval,
)

print(f"Split: Train={len(X_train)}, Val={len(X_val)}, Test={len(X_test)}")

# Poids de classes pour l'ethnicité
ethnicity_weights = compute_class_weight(
    "balanced", classes=np.unique(y_eth_train), y=y_eth_train
)
ethnicity_class_weights = dict(enumerate(ethnicity_weights))
print(f"\nPoids ethnicité: {ethnicity_class_weights}")

class_weights_tensor = tf.constant(
    [ethnicity_class_weights[i] for i in range(NUM_ETHNICITY_CLASSES)], dtype=tf.float32
)

Split: Train=16543, Val=3545, Test=3545

Poids ethnicité: {0: np.float64(0.4710421412300683), 1: np.float64(1.045701643489254), 2: np.float64(1.3837724801338351), 3: np.float64(1.1901438848920862), 4: np.float64(2.794425675675676)}


I0000 00:00:1769071409.140688   18715 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 4130 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


## 5. Focal Loss


In [ ]:
class FocalLoss(keras.losses.Loss):
    """Focal Loss pour classification multi-classe."""

    def __init__(self, gamma=2.0, class_weights=None, name="focal_loss", **kwargs):
        super().__init__(name=name, **kwargs)
        self.gamma = gamma
        self.class_weights = class_weights

    def call(self, y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)
        y_true = tf.cast(y_true, tf.int32)
        y_true_one_hot = tf.one_hot(tf.squeeze(y_true), depth=tf.shape(y_pred)[-1])

        ce = -y_true_one_hot * tf.math.log(y_pred)
        p_t = tf.reduce_sum(y_true_one_hot * y_pred, axis=-1, keepdims=True)
        focal_weight = tf.pow(1 - p_t, self.gamma)

        if self.class_weights is not None:
            weights = tf.reduce_sum(
                y_true_one_hot * self.class_weights, axis=-1, keepdims=True
            )
            focal_weight = focal_weight * weights

        focal_loss = focal_weight * ce
        return tf.reduce_mean(tf.reduce_sum(focal_loss, axis=-1))

    def get_config(self):
        config = super().get_config()
        config.update(
            {
                "gamma": self.gamma,
                "class_weights": (
                    self.class_weights.numpy().tolist()
                    if self.class_weights is not None
                    else None
                ),
            }
        )
        return config

    @classmethod
    def from_config(cls, config):
        class_weights = config.pop("class_weights", None)
        if class_weights is not None:
            class_weights = tf.constant(class_weights, dtype=tf.float32)
        return cls(class_weights=class_weights, **config)


print("Focal Loss définie")

Focal Loss définie


## 6. Data Augmentation


In [6]:
data_augmentation = keras.Sequential(
    [
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.1),
        layers.RandomZoom(0.15),
        layers.RandomTranslation(0.1, 0.1),
        layers.RandomBrightness(0.2),
        layers.RandomContrast(0.2),
    ],
    name="data_augmentation",
)

print("Data augmentation configurée")

Data augmentation configurée


## 7. Pipeline de Données


In [7]:
def load_and_preprocess(filepath, age, gender, ethnicity):
    img = tf.io.read_file(filepath)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.cast(img, tf.float32)
    return img, {
        "age_output": age,
        "gender_output": gender,
        "ethnicity_output": ethnicity,
    }


def create_dataset(filepaths, ages, genders, ethnicities, training=False):
    ds = tf.data.Dataset.from_tensor_slices((filepaths, ages, genders, ethnicities))
    ds = ds.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)

    if training:
        ds = ds.shuffle(buffer_size=2000)

    ds = ds.batch(BATCH_SIZE)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds


train_ds = create_dataset(
    X_train, y_age_train, y_gender_train, y_eth_train, training=True
)
val_ds = create_dataset(X_val, y_age_val, y_gender_val, y_eth_val, training=False)
test_ds = create_dataset(X_test, y_age_test, y_gender_test, y_eth_test, training=False)

print(f"Datasets créés")

Datasets créés


## 8. Architecture avec MobileNetV3

### Le principe du Transfer Learning expliqué

**MobileNet a été entraîné sur ImageNet** (1.2 millions d'images, 1000 classes).

Il a appris à reconnaître :

- Couches basses : edges, textures, couleurs
- Couches moyennes : formes, patterns
- Couches hautes : objets, parties du corps, visages basiques

**On réutilise cette connaissance** sans la modifier (backbone gelé), et on ajoute seulement nos propres couches pour nos 3 tâches spécifiques.


In [ ]:
def build_mobilenet_multitask_model(input_shape=(IMG_SIZE, IMG_SIZE, 3), training=True):
    """
    Modèle multi-tâche avec MobileNetV3 comme backbone.

    Le backbone est PRÉ-ENTRAÎNÉ sur ImageNet et GELÉ.
    On entraîne seulement nos couches personnalisées.
    """

    inputs = layers.Input(shape=input_shape, name="input_image")

    # Data augmentation (seulement en training)
    x = data_augmentation(inputs) if training else inputs

    # =============================================
    # BACKBONE MOBILENETV3 (PRÉ-ENTRAÎNÉ, GELÉ)
    # =============================================
    # - weights="imagenet" : charge les poids pré-entraînés
    # - include_top=False : enlève la dernière couche (1000 classes)
    # - trainable=False : on ne touche PAS aux poids

    if USE_LARGE:
        backbone = MobileNetV3Large(
            include_top=False,
            weights="imagenet",
            input_tensor=x,
            include_preprocessing=True,  # MobileNet normalise en interne
        )
    else:
        backbone = MobileNetV3Small(
            include_top=False,
            weights="imagenet",
            input_tensor=x,
            include_preprocessing=True,
        )

    # GELER le backbone - on ne modifie pas ses poids
    backbone.trainable = False

    print(f"\n🔒 Backbone MobileNetV3{'Large' if USE_LARGE else 'Small'} GELÉ")
    print(f"   Paramètres backbone: {backbone.count_params():,}")

    # Extraire les features
    features = backbone.output

    # Global Average Pooling
    pooled = layers.GlobalAveragePooling2D(name="gap")(features)

    # =============================================
    # NOS COUCHES PERSONNALISÉES (À ENTRAÎNER)
    # =============================================

    # Couche partagée
    shared = layers.Dense(512, kernel_regularizer=l2(1e-4), name="shared_dense")(pooled)
    shared = layers.BatchNormalization(name="shared_bn")(shared)
    shared = layers.Activation("relu")(shared)
    shared = layers.Dropout(0.4, name="shared_dropout")(shared)

    # === HEAD ÂGE ===
    age_branch = layers.Dense(128, kernel_regularizer=l2(1e-4), name="age_dense")(
        shared
    )
    age_branch = layers.BatchNormalization(name="age_bn")(age_branch)
    age_branch = layers.Activation("relu")(age_branch)
    age_branch = layers.Dropout(0.3, name="age_dropout")(age_branch)
    age_output = layers.Dense(1, activation="linear", name="age_output")(age_branch)

    # === HEAD GENRE ===
    gender_branch = layers.Dense(128, kernel_regularizer=l2(1e-4), name="gender_dense")(
        shared
    )
    gender_branch = layers.BatchNormalization(name="gender_bn")(gender_branch)
    gender_branch = layers.Activation("relu")(gender_branch)
    gender_branch = layers.Dropout(0.3, name="gender_dropout")(gender_branch)
    gender_output = layers.Dense(1, activation="sigmoid", name="gender_output")(
        gender_branch
    )

    # === HEAD ETHNICITÉ ===
    eth_branch = layers.Dense(128, kernel_regularizer=l2(1e-4), name="eth_dense")(
        shared
    )
    eth_branch = layers.BatchNormalization(name="eth_bn")(eth_branch)
    eth_branch = layers.Activation("relu")(eth_branch)
    eth_branch = layers.Dropout(0.3, name="eth_dropout")(eth_branch)
    ethnicity_output = layers.Dense(
        NUM_ETHNICITY_CLASSES, activation="softmax", name="ethnicity_output"
    )(eth_branch)

    model = Model(
        inputs=inputs,
        outputs=[age_output, gender_output, ethnicity_output],
        name="mobilenet_multitask_model",
    )

    return model, backbone


model, backbone = build_mobilenet_multitask_model(training=True)

# Compter les paramètres
total_params = model.count_params()
trainable_params = sum(
    [tf.keras.backend.count_params(w) for w in model.trainable_weights]
)
non_trainable_params = total_params - trainable_params

print(f"\n📊 Résumé des paramètres:")
print(f"   Total: {total_params:,}")
print(f"   Entraînables: {trainable_params:,} (nos couches)")
print(f"   Non-entraînables: {non_trainable_params:,} (MobileNet gelé)")
print(f"   Ratio: {trainable_params/total_params*100:.1f}% à entraîner")

12683000/12683000 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step

🔒 Backbone MobileNetV3Large GELÉ
   Paramètres backbone: 2,996,352

📊 Résumé des paramètres:
   Total: 3,689,863
   Entraînables: 691,719 (nos couches)
   Non-entraînables: 2,998,144 (MobileNet gelé)
   Ratio: 18.7% à entraîner


In [9]:
model.summary()

Model: "mobilenet_multitask_model"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_image         │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ data_augmentation   │ (None, 128, 128,  │          0 │ input_image[0][0] │
│ (Sequential)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling           │ (None, 128, 128,  │          0 │ data_augmentatio… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv (Conv2D)       │ (None, 64, 64,    │        432 │ rescaling[0][0]   │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_bn             │ (None, 64, 64,    │         64 │ conv[0][0]        │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 64, 64,    │          0 │ conv_bn[0][0]     │
│ (Activation)        │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 64, 64,    │        144 │ activation[0][0]  │
│ (DepthwiseConv2D)   │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 64, 64,    │         64 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu (ReLU)        │ (None, 64, 64,    │          0 │ expanded_conv_de… │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 64, 64,    │        256 │ re_lu[0][0]       │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 64, 64,    │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_add   │ (None, 64, 64,    │          0 │ activation[0][0], │
│ (Add)               │ 16)               │            │ expanded_conv_pr… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_1_ex… │ (None, 64, 64,    │      1,024 │ expanded_conv_ad… │
│ (Conv2D)            │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_1_ex… │ (None, 64, 64,    │        256 │ expanded_conv_1_… │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_1 (ReLU)      │ (None, 64, 64,    │          0 │ expanded_conv_1_… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_1_de… │ (None, 65, 65,    │          0 │ re_lu_1[0][0]     │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_1_de… │ (None, 32, 32,    │        576 │ expanded_conv_1_

 Total params: 3,689,863 (14.08 MB)

 Trainable params: 691,719 (2.64 MB)

 Non-trainable params: 2,998,144 (11.44 MB)

## 9. Compilation


In [10]:
steps_per_epoch = len(X_train) // BATCH_SIZE
total_warmup_steps = steps_per_epoch * WARMUP_EPOCHS

warmup_lr_schedule = CosineDecay(
    initial_learning_rate=INITIAL_LR,
    decay_steps=total_warmup_steps,
    alpha=0.01,
)

losses = {
    "age_output": keras.losses.Huber(delta=3.0),
    "gender_output": keras.losses.BinaryCrossentropy(label_smoothing=0.1),
    "ethnicity_output": FocalLoss(gamma=2.0, class_weights=class_weights_tensor),
}

loss_weights = {
    "age_output": 1.5,
    "gender_output": 1.0,
    "ethnicity_output": 2.0,
}

metrics = {
    "age_output": ["mae", "mse"],
    "gender_output": ["accuracy", keras.metrics.AUC(name="auc")],
    "ethnicity_output": ["accuracy"],
}

model.compile(
    optimizer=AdamW(learning_rate=warmup_lr_schedule, weight_decay=1e-5),
    loss=losses,
    loss_weights=loss_weights,
    metrics=metrics,
)

print("Modèle compilé avec:")
print(f"  - Backbone: MobileNetV3 (GELÉ)")
print(f"  - Loss âge: Huber (δ=3.0)")
print(f"  - Loss genre: BCE + Label Smoothing")
print(f"  - Loss ethnicité: Focal Loss + Class Weights")

Modèle compilé avec:
  - Backbone: MobileNetV3 (GELÉ)
  - Loss âge: Huber (δ=3.0)
  - Loss genre: BCE + Label Smoothing
  - Loss ethnicité: Focal Loss + Class Weights


## 10. Phase 1: Warmup (Backbone Gelé)

Dans cette phase, **MobileNet reste complètement gelé**.
On entraîne uniquement nos couches personnalisées.


In [11]:
warmup_callbacks = [
    callbacks.ModelCheckpoint(
        filepath=str(ARTIFACTS_DIR / "mobilenet_warmup_best.keras"),
        monitor="val_loss",
        save_best_only=True,
        verbose=1,
    ),
    callbacks.EarlyStopping(
        monitor="val_loss",
        patience=8,
        restore_best_weights=True,
        verbose=1,
    ),
]

print("=" * 60)
print("PHASE 1: WARMUP (Backbone MobileNet GELÉ)")
print("=" * 60)
print(f"Paramètres entraînables: {trainable_params:,}")

warmup_history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=WARMUP_EPOCHS,
    callbacks=warmup_callbacks,
    verbose=1,
)

PHASE 1: WARMUP (Backbone MobileNet GELÉ)
Paramètres entraînables: 691,719
Epoch 1/15
516/517 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - age_output_loss: 67.4705 - age_output_mae: 23.9318 - age_output_mse: 907.0380 - ethnicity_output_accuracy: 0.3763 - ethnicity_output_loss: 1.0951 - gender_output_accuracy: 0.7112 - gender_output_auc: 0.7883 - gender_output_loss: 0.6071 - loss: 104.1406
Epoch 1: val_loss improved from None to 35.15343, saving model to artifacts/mobilenet_warmup_best.keras

Epoch 1: finished saving model to artifacts/mobilenet_warmup_best.keras
517/517 ━━━━━━━━━━━━━━━━━━━━ 25s 30ms/step - age_output_loss: 47.3591 - age_output_mae: 17.1882 - age_output_mse: 551.6234 - ethnicity_output_accuracy: 0.4164 - ethnicity_output_loss: 0.9533 - gender_output_accuracy: 0.7380 - gender_output_auc: 0.8147 - gender_output_loss: 0.5724 - loss: 73.6676 - val_age_output_loss: 22.0277 - val_age_output_mae: 8.6706 - val_age_output_mse: 150.1648 - val_ethnicity_output_accuracy: 0.5475 - val_ethnic

## 11. Phase 2: Fine-tuning (Dégel partiel)

On dégèle les **dernières couches** de MobileNet pour affiner les features spécifiques aux visages.


In [ ]:
print("=" * 60)
print("PHASE 2: FINE-TUNING (Dégel partiel du backbone)")
print("=" * 60)

# Dégeler les dernières couches
backbone.trainable = True

# Geler toutes les couches sauf les 50 dernières
freeze_until = len(backbone.layers) - 50
for layer in backbone.layers[:freeze_until]:
    layer.trainable = False

# Compter les nouveaux paramètres entraînables
trainable_params_ft = sum(
    [tf.keras.backend.count_params(w) for w in model.trainable_weights]
)
print(f"Paramètres entraînables après dégel: {trainable_params_ft:,}")

# Nouveau LR schedule (plus bas car on touche au backbone)
total_finetune_steps = steps_per_epoch * FINETUNE_EPOCHS
finetune_lr_schedule = CosineDecay(
    initial_learning_rate=FINETUNE_LR,
    decay_steps=total_finetune_steps,
    alpha=0.001,
)

model.compile(
    optimizer=AdamW(learning_rate=finetune_lr_schedule, weight_decay=1e-5),
    loss=losses,
    loss_weights=loss_weights,
    metrics=metrics,
)

finetune_callbacks = [
    callbacks.ModelCheckpoint(
        filepath=str(ARTIFACTS_DIR / "mobilenet_model_best.keras"),
        monitor="val_loss",
        save_best_only=True,
        verbose=1,
    ),
    callbacks.EarlyStopping(
        monitor="val_loss",
        patience=12,
        restore_best_weights=True,
        verbose=1,
    ),
]

finetune_history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=FINETUNE_EPOCHS,
    callbacks=finetune_callbacks,
    verbose=1,
)

PHASE 2: FINE-TUNING (Dégel partiel du backbone)
Paramètres entraînables après dégel: 2,871,407
Epoch 1/40
517/517 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step - age_output_loss: 30.3166 - age_output_mae: 11.4983 - age_output_mse: 249.1388 - ethnicity_output_accuracy: 0.4780 - ethnicity_output_loss: 0.8003 - gender_output_accuracy: 0.7256 - gender_output_auc: 0.8569 - gender_output_loss: 0.5751 - loss: 47.9162
Epoch 1: val_loss improved from None to 34.86915, saving model to artifacts/mobilenet_model_best.keras

Epoch 1: finished saving model to artifacts/mobilenet_model_best.keras
517/517 ━━━━━━━━━━━━━━━━━━━━ 47s 59ms/step - age_output_loss: 28.8334 - age_output_mae: 11.0018 - age_output_mse: 225.8681 - ethnicity_output_accuracy: 0.4791 - ethnicity_output_loss: 0.7970 - gender_output_accuracy: 0.7286 - gender_output_auc: 0.8331 - gender_output_loss: 0.5702 - loss: 45.6798 - val_age_output_loss: 21.8845 - val_age_output_mae: 8.6266 - val_age_output_mse: 148.5226 - val_ethnicity_output_accuracy: 0

## 12. Évaluation


In [13]:
# Charger le meilleur modèle
best_model = keras.models.load_model(
    ARTIFACTS_DIR / "mobilenet_model_best.keras",
    custom_objects={"FocalLoss": FocalLoss},
)

# Prédictions
predictions = best_model.predict(test_ds, verbose=1)
y_pred_age = predictions[0].flatten()
y_pred_gender = (predictions[1].flatten() > 0.5).astype(int)
y_pred_ethnicity = np.argmax(predictions[2], axis=1)

# Métriques
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    mean_absolute_error,
)

mae_age = mean_absolute_error(y_age_test, y_pred_age)
rmse_age = np.sqrt(np.mean((y_age_test - y_pred_age) ** 2))
acc_gender = accuracy_score(y_gender_test, y_pred_gender)
f1_gender = f1_score(y_gender_test, y_pred_gender)
acc_ethnicity = accuracy_score(y_eth_test, y_pred_ethnicity)
f1_ethnicity_macro = f1_score(y_eth_test, y_pred_ethnicity, average="macro")

print("\n" + "=" * 60)
print("RÉSULTATS MOBILENETV3")
print("=" * 60)
print(f"\n🎂 ÂGE:")
print(f"   MAE: {mae_age:.2f} années")
print(f"   RMSE: {rmse_age:.2f} années")
print(f"\n👫 GENRE:")
print(f"   Accuracy: {acc_gender*100:.2f}%")
print(f"   F1-Score: {f1_gender:.4f}")
print(f"\n🌍 ETHNICITÉ:")
print(f"   Accuracy: {acc_ethnicity*100:.2f}%")
print(f"   F1-Score (macro): {f1_ethnicity_macro:.4f}")

111/111 ━━━━━━━━━━━━━━━━━━━━ 3s 20ms/step

RÉSULTATS MOBILENETV3

🎂 ÂGE:
   MAE: 6.47 années
   RMSE: 9.20 années

👫 GENRE:
   Accuracy: 83.58%
   F1-Score: 0.8186

🌍 ETHNICITÉ:
   Accuracy: 62.45%
   F1-Score (macro): 0.5605


In [ ]:
print("\nRAPPORT - GENRE")
print(classification_report(y_gender_test, y_pred_gender, target_names=GENDER_CLASSES))

print("\nRAPPORT - ETHNICITÉ")
print(
    classification_report(y_eth_test, y_pred_ethnicity, target_names=ETHNICITY_CLASSES)
)


RAPPORT - GENRE
              precision    recall  f1-score   support

        Male       0.81      0.89      0.85      1851
      Female       0.87      0.78      0.82      1694

    accuracy                           0.84      3545
   macro avg       0.84      0.83      0.83      3545
weighted avg       0.84      0.84      0.84      3545


RAPPORT - ETHNICITÉ
              precision    recall  f1-score   support

       White       0.83      0.67      0.74      1505
       Black       0.69      0.78      0.73       678
       Asian       0.58      0.64      0.61       512
      Indian       0.56      0.40      0.46       596
      Others       0.18      0.40      0.25       254

    accuracy                           0.62      3545
   macro avg       0.57      0.58      0.56      3545
weighted avg       0.67      0.62      0.64      3545



## 13. Sauvegarde et Conversion TFLite


In [15]:
# Sauvegarder le modèle final
final_model_path = ARTIFACTS_DIR / "mobilenet_model_final.keras"
best_model.save(final_model_path)
print(f"Modèle sauvegardé: {final_model_path}")

# Conversion TFLite
converter = tf.lite.TFLiteConverter.from_keras_model(best_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]

tflite_model = converter.convert()

tflite_path = ARTIFACTS_DIR / "mobilenet_model.tflite"
with open(tflite_path, "wb") as f:
    f.write(tflite_model)

tflite_size_mb = tflite_path.stat().st_size / (1024 * 1024)
print(f"TFLite sauvegardé: {tflite_path} ({tflite_size_mb:.2f} MB)")

# Comparaison avec EfficientNet
print(f"\n📊 Comparaison taille TFLite:")
print(f"   MobileNetV3: {tflite_size_mb:.2f} MB")
print(f"   EfficientNetB0: ~15 MB")
print(f"   Réduction: ~{(1 - tflite_size_mb/15)*100:.0f}%")

Modèle sauvegardé: artifacts/mobilenet_model_final.keras
INFO:tensorflow:Assets written to: /tmp/tmp0f4quljx/assets


INFO:tensorflow:Assets written to: /tmp/tmp0f4quljx/assets


Saved artifact at '/tmp/tmp0f4quljx'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 128, 128, 3), dtype=tf.float32, name='input_image')
Output Type:
  List[TensorSpec(shape=(None, 1), dtype=tf.float32, name=None), TensorSpec(shape=(None, 1), dtype=tf.float32, name=None), TensorSpec(shape=(None, 5), dtype=tf.float32, name=None)]
Captures:
  128325082703312: TensorSpec(shape=(), dtype=tf.resource, name=None)
  128325082705232: TensorSpec(shape=(), dtype=tf.resource, name=None)
  128325082706384: TensorSpec(shape=(), dtype=tf.resource, name=None)
  128325082701008: TensorSpec(shape=(), dtype=tf.resource, name=None)
  128325082708688: TensorSpec(shape=(), dtype=tf.resource, name=None)
  128325082706000: TensorSpec(shape=(), dtype=tf.resource, name=None)
  128325082703888: TensorSpec(shape=(), dtype=tf.resource, name=None)
  128325082705040: TensorSpec(shape=(), dtype=tf.resource, name=None)
  128325082706768: TensorSpec(shape

W0000 00:00:1769072674.118542   18715 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1769072674.118583   18715 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1769072674.242013   18715 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled


## 14. Métadonnées


In [16]:
import json

model_info = {
    "model_type": "mobilenet_multitask",
    "version": "1.0",
    "backbone": f"MobileNetV3{'Large' if USE_LARGE else 'Small'}",
    "img_size": IMG_SIZE,
    "input_range": [0, 255],
    "transfer_learning": {
        "pretrained_on": "ImageNet",
        "backbone_frozen_initially": True,
        "fine_tuned_layers": 50,
    },
    "outputs": {
        "age": {"type": "regression", "output_name": "age_output"},
        "gender": {
            "type": "binary_classification",
            "output_name": "gender_output",
            "classes": GENDER_CLASSES,
            "threshold": 0.5,
        },
        "ethnicity": {
            "type": "multiclass_classification",
            "output_name": "ethnicity_output",
            "classes": ETHNICITY_CLASSES,
        },
    },
    "metrics": {
        "age_mae": float(mae_age),
        "age_rmse": float(rmse_age),
        "gender_accuracy": float(acc_gender),
        "gender_f1": float(f1_gender),
        "ethnicity_accuracy": float(acc_ethnicity),
        "ethnicity_f1_macro": float(f1_ethnicity_macro),
    },
}

info_path = ARTIFACTS_DIR / "mobilenet_model_info.json"
with open(info_path, "w") as f:
    json.dump(model_info, f, indent=2)

print(f"Métadonnées sauvegardées: {info_path}")
print(json.dumps(model_info, indent=2))

Métadonnées sauvegardées: artifacts/mobilenet_model_info.json
{
  "model_type": "mobilenet_multitask",
  "version": "1.0",
  "backbone": "MobileNetV3Large",
  "img_size": 128,
  "input_range": [
    0,
    255
  ],
  "transfer_learning": {
    "pretrained_on": "ImageNet",
    "backbone_frozen_initially": true,
    "fine_tuned_layers": 50
  },
  "outputs": {
    "age": {
      "type": "regression",
      "output_name": "age_output"
    },
    "gender": {
      "type": "binary_classification",
      "output_name": "gender_output",
      "classes": [
        "Male",
        "Female"
      ],
      "threshold": 0.5
    },
    "ethnicity": {
      "type": "multiclass_classification",
      "output_name": "ethnicity_output",
      "classes": [
        "White",
        "Black",
        "Asian",
        "Indian",
        "Others"
      ]
    }
  },
  "metrics": {
    "age_mae": 6.4678778648376465,
    "age_rmse": 9.201828956604004,
    "gender_accuracy": 0.8358251057827927,
    "gender_f1": 0.

## 15. Résumé Final


In [ ]:
print("=" * 60)
print("RÉSUMÉ MODÈLE MOBILENETV3 MULTI-TÂCHE")
print("=" * 60)
print(f"\n📊 Architecture:")
print(
    f"   - Backbone: MobileNetV3{'Large' if USE_LARGE else 'Small'} (pré-entraîné ImageNet)"
)
print(f"   - Transfer Learning: Oui (backbone gelé puis fine-tuné)")
print(f"   - Couches personnalisées: 512 → 128 par tâche")
print(f"\n📈 Performances:")
print(f"   🎂 Âge: MAE={mae_age:.2f}, RMSE={rmse_age:.2f}")
print(f"   👫 Genre: Acc={acc_gender*100:.2f}%, F1={f1_gender:.4f}")
print(f"   🌍 Ethnicité: Acc={acc_ethnicity*100:.2f}%, F1={f1_ethnicity_macro:.4f}")
print(f"\n💾 Fichiers:")
print(f"   - {final_model_path}")
print(f"   - {tflite_path} ({tflite_size_mb:.2f} MB)")
print("=" * 60)

RÉSUMÉ MODÈLE MOBILENETV3 MULTI-TÂCHE

📊 Architecture:
   - Backbone: MobileNetV3Large (pré-entraîné ImageNet)
   - Transfer Learning: Oui (backbone gelé puis fine-tuné)
   - Couches personnalisées: 512 → 128 par tâche

📈 Performances:
   🎂 Âge: MAE=6.47, RMSE=9.20
   👫 Genre: Acc=83.58%, F1=0.8186
   🌍 Ethnicité: Acc=62.45%, F1=0.5605

💾 Fichiers:
   - artifacts/mobilenet_model_final.keras
   - artifacts/mobilenet_model.tflite (7.03 MB)
